# Dự đoán thao tác chỉnh sửa đồ thị cảnh

Chạy **Restart Kernel and Run All Cells** với môi trường `myenv` của dự án. Notebook dùng checkpoint, trọng số CLIP, cache văn bản và dữ liệu đã có trong workspace. Nếu thiếu manifest VN100, chạy `bash scripts/package_level3.sh prepare` từ thư mục gốc.

Ba ví dụ được chọn để minh họa ba mức độ đúng. Mỗi ca ghi seed của checkpoint GNN MagicBrush nhánh prompt. Trong cả ba ca, GNN chạy trực tiếp trên CPU với hai đầu vào là đồ thị nguồn và câu yêu cầu.

Ở hai ca VN100, Qwen đã chỉnh ảnh độc lập theo cùng câu yêu cầu. Notebook đọc ảnh Qwen cùng các đồ thị REACT++ đã lưu và tính lại điểm cho cặp ảnh. **E** đo mức đáp ứng yêu cầu, **P** đo bảo toàn phần còn lại và **G = E × P** khi đủ bằng chứng. Khung đỏ là đối tượng phát hiện được, khung vàng là vùng chỉnh sửa dự kiến.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "tools/graph_editing_live_demo_v1.py").is_file())
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
from IPython.display import HTML, Image, display
from src.editing.operations import EditGraph
from tools.graph_editing_live_demo_v1 import predict
from tools.level3_submission_v1.demo import checked, graph_html, image_pair_html, open_bundle, recompute, table

torch.set_num_threads(2)
root, manifest = open_bundle(ROOT)

In [ ]:
def show_source(task_id):
    case = next(c for c in manifest["cases"] if c["task_id"] == task_id)
    print(case["task_id"], case["instruction"], sep="\n")
    display(Image(filename=str(checked(root, manifest, case["source"])), width=520))
    source_record = json.loads(checked(root, manifest, case["graphs"]["source"]).read_text())
    source = EditGraph.from_dict(source_record["source_graph"])
    display(HTML(graph_html(source.to_dict(), "Đồ thị nguồn do REACT++ dự đoán", task_id + "-source")))
    return case, source


def show_prediction(case, source, seed):
    operations, predicted, selection = predict(source, case["instruction"], seed=seed)
    print(f"GNN MagicBrush · prompt · seed {selection['seed']} · epoch {selection['selected_epoch']}")
    print("Checkpoint SHA-256:", selection["sha256"])
    print("Thao tác dự đoán:", json.dumps([op.to_dict() for op in operations], ensure_ascii=False, indent=2))
    display(HTML(graph_html(predicted.to_dict(), "Đồ thị đích dựng từ thao tác GNN", case["task_id"] + "-gnn")))


def show_score(value):
    return "Không chấm được" if value is None else f"{value:.3f}"


def show_image_evaluation(case):
    scored = recompute(root, manifest, case["task_id"])
    display(HTML(image_pair_html(root, scored)))
    display(HTML(graph_html(scored["graphs_loaded"]["target"], "Đồ thị trích từ ảnh Qwen", case["task_id"] + "-qwen")))
    s, labels = scored["recomputed"], scored["scores_and_labels"]


    display(HTML(table(["E", "P", "G", "Nhãn tay: đúng yêu cầu", "Nhãn tay: bảo toàn"],
                       [[show_score(s["edit_score"]), show_score(s["preservation_score"]),
                         show_score(s["graph_consistency"]), labels["edit_fulfillment"], labels["preservation"]]])))
    print("Điều kiện chỉnh sửa:", s.get("checks", []))
    print("P đủ bằng chứng:", s["preservation_evaluable"])
    print("Lý do hạn chế:", s["reason_codes"] or "—")

## 1. Thêm thuyền buồm · VN100 · seed 42

Ảnh nguồn đã có một chiếc thuyền. Câu yêu cầu đề nghị thêm một chiếc thuyền buồm nhỏ ở tiền cảnh.

In [ ]:
boat, boat_source = show_source("travel-7bb0c6b3635205-q100-1")
show_prediction(boat, boat_source, seed=42)

In [ ]:
show_image_evaluation(boat)

GNN thêm nút `giraffe`, sai đối tượng. Trên ảnh Qwen, bộ chấm phát hiện số thuyền tăng từ một lên hai, với chiếc mới trong vùng yêu cầu. Cặp ảnh đạt G = 1.

## 2. Thêm chó trên đá · VN100 · seed 3407

Yêu cầu gồm cả đối tượng và vị trí. Hãy tìm nút `rock` trong đồ thị nguồn, sau đó kiểm tra nút nào được thêm và có cạnh nào nối nó với đá.

In [ ]:
dog, dog_source = show_source("travel-a4fe5a686b0396-q200-2")

In [ ]:
show_prediction(dog, dog_source, seed=3407)

GNN thêm đúng một nút `dog` và giữ nguyên các nút, cạnh cũ. Nút mới đứng riêng nên đồ thị chỉ thể hiện được một phần yêu cầu. Chi tiết chó đứng trên đá ở tiền cảnh cạnh hồ chưa được mô hình mã hóa.

In [ ]:
show_image_evaluation(dog)

Trong ảnh Qwen, bộ chấm xác nhận có thêm chó trong vùng yêu cầu (E = 1). Phần đồ thị còn lại có P ≈ 0,829 nên G ≈ 0,829. Người duyệt đánh dấu đạt cả chỉnh sửa lẫn bảo toàn.

## 3. Xóa ghế · MagicBrush test · seed 123

Ca `magicbrush:190052:2` được chọn để minh họa một dự đoán khớp hoàn toàn. Với checkpoint GNN nhánh prompt, seed 123, có **1/59** ca test đạt tiêu chí này. Hãy nhìn vào ảnh và cạnh nối ghế với quầy trước khi chạy ô dự đoán.

In [ ]:
from experiments.magicbrush_prompt_intent_v1.dataset import rows
from experiments.magicbrush_prompt_pilot_v1.run import example
from tools.emnlp2020_graph_modification_v1.dataset import sha256

chair = next(row for row in rows("test") if row["id"] == "magicbrush:190052:2")

def show_magicbrush_image(role):
    checksum = chair["images"][role]
    path = ROOT / "data/processed/level2/magicbrush-v1/images" / f"{checksum}.png"
    if sha256(path) != checksum:
        raise ValueError(f"Ảnh sai checksum: {role}")
    display(Image(filename=str(path), width=460))

print(chair["id"], chair["instruction"], sep="\n")
show_magicbrush_image("source_image")
chairs_source = EditGraph.from_dict(chair["source_graph"])
display(HTML(graph_html(chairs_source.to_dict(), "Đồ thị nguồn MagicBrush", "chair-source")))

In [ ]:
chair_ops, chair_graph, chair_checkpoint = predict(chairs_source, chair["instruction"], seed=123)
print("Checkpoint:", chair_checkpoint["checkpoint"])
print("Thao tác GNN:", json.dumps([op.to_dict() for op in chair_ops], ensure_ascii=False, indent=2))
display(HTML(graph_html(chair_graph.to_dict(), "Đồ thị sau khi áp dụng dự đoán", "chair-pred")))

In [ ]:
reference = example(chair, chair["operations"])
print("Khớp toàn bộ thao tác:", set(chair_ops) == set(reference.operations))
print("Khớp đồ thị đích:", chair_graph == reference.target)
print("Ảnh đích tham chiếu của MagicBrush")
show_magicbrush_image("target_image")

GNN xóa nút `chairs` và cạnh `chairs → in front of → counter`. Các nút và cạnh khác được giữ lại. Ảnh đích bên trên thuộc cặp dữ liệu MagicBrush.